# blog datas analyze
reference: https://keras.io/examples/nlp/multi_label_classification/

In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval

In [2]:
# CSV 파일 불러오기
data = pd.read_csv("data/blog_tags_result.csv")

# 태그를 쉼표로 분리하여 리스트로 변환
data["tags"] = data["tags"].apply(lambda x: [tag.strip() for tag in str(x).split(',')])

data.head()

,link,title,tags
0,https://blog.banksalad.com/pnc/culture-faq/,"뱅크샐러드 채용, 무엇이든 물어보세요",[홍보]
1,https://blog.banksalad.com/pnc/design_dressing/,제1회 뱅크샐러드 디자인 드레싱을 소개합니다,[디자인]
2,https://blog.banksalad.com/pnc/mydata-handbook...,마이데이터 맵과 비즈니스 확장성,"[보안, 기획]"
3,https://blog.banksalad.com/pnc/hackathon_2022/,뱅크샐러드 사내 해커톤을 소개합니다!,[개발문화]
4,https://blog.banksalad.com/pnc/office-tour/,뱅크샐러드 오피스 투어 [업무공간 편],[홍보]


In [3]:
print(f"데이터셋에 {len(data)} 열이 있습니다")

데이터셋에 1092 열이 있습니다


In [4]:
total_duplicate_titles = sum(data["title"].duplicated())
print(f"중복 제목이 {total_duplicate_titles} 개 있습니다.")

중복 제목이 2 개 있습니다.


In [5]:
# 중복 제거
data = data[~data["title"].duplicated()]
print(f"중복 제거 후 데이터셋 행 수: {len(data)}")

# 모든 태그를 펼쳐서 빈도수 계산
from collections import Counter

all_tags = []
for tags in data["tags"]:
    all_tags.extend(tags)

# 개별 태그 빈도수 계산
tag_counts = Counter(all_tags)

# 1번만 등장하는 태그 개수
single_occurrence_tags = sum(1 for count in tag_counts.values() if count == 1)
print(f"1번만 등장하는 태그 수: {single_occurrence_tags}개")

# 고유 태그 수
print(f"총 고유 태그 수: {len(tag_counts)}개")
print(f"총 태그 발생 횟수: {len(all_tags)}회")

# 모든 태그 확인
print("\n=== 모든 태그 ===")
for tag, count in tag_counts.most_common():
    print(f"{tag}: {count}회")

중복 제거 후 데이터셋 행 수: 1090
1번만 등장하는 태그 수: 0개
총 고유 태그 수: 14개
총 태그 발생 횟수: 1951회

=== 모든 태그 ===
백엔드: 298회
프론트엔드: 296회
DevOps: 285회
컨퍼런스: 216회
데이터: 201회
모바일: 171회
AI: 141회
테스트: 92회
보안: 65회
개발문화: 63회
디자인: 58회
홍보: 31회
기획: 29회
기타: 5회


# multi-label text classification

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, ops

from sklearn.model_selection import train_test_split

from ast import literal_eval
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [7]:
# 처음 5개 데이터의 tags 확인
data["tags"].values[:5]

array([list(['홍보']), list(['디자인']), list(['보안', '기획']), list(['개발문화']),
       list(['홍보'])], dtype=object)

In [14]:
value_counts = data["tags"].value_counts()
valid_tags = value_counts[value_counts >= 2].index
filtered = data[data["tags"].isin(valid_tags)]

## 클래스 불균형을 고려하여 계층적 분할(stratified split)을 사용합니다

In [ ]:
test_split = 0.1

# Initial train and test split.
train_df, test_df = train_test_split(
    filtered,
    test_size=test_split,
    stratify=filtered["tags"],
)

# Splitting the test set further into validation
# and new test sets.
val_df = test_df.sample(frac=0.5)
test_df.drop(val_df.index, inplace=True)

print(f"Number of rows in training set: {len(train_df)}")
print(f"Number of rows in validation set: {len(val_df)}")
print(f"Number of rows in test set: {len(test_df)}")

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.